# 04 · Top metapaths: Gandalf vs local ARAGORN

Same dataset and **treats-only mask**: held-out direct `treats` edges are removed; other direct predicates remain.

Ranked by **distinct expected chemical–disease pairs recovered**, not pattern frequency. Counts overlap across rows. Recovery means the chemical appears in the retrieved graph, not ranking accuracy.


In [1]:
SPLIT = 'validation'  # 'test' is exploratory.
TOP_N = 5


In [2]:
from pathlib import Path
import json, hashlib, html
import pandas as pd
from IPython.display import display, HTML, Markdown

ROOT = next(p / 'experiments/ultra' for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'experiments/ultra/download.json').exists())
OUT = ROOT / 'pilot/results/chemical_metapaths_001'
WALKS = ROOT / 'pilot/results/chemical_walks_treats_only_001'
read = lambda p: json.loads(p.read_text())
assert SPLIT in ('validation','test') and isinstance(TOP_N,int) and TOP_N > 0
manifest = read(OUT / 'manifest.json')
for name in ('answer_metapaths.csv','rule_catalog.json'):
    assert hashlib.sha256((OUT / name).read_bytes()).hexdigest() == manifest['outputs'][name]
assert hashlib.sha256((WALKS / 'summary.json').read_bytes()).hexdigest() == manifest['inputs'][str((WALKS / 'summary.json').relative_to(ROOT))]
comparison = read(WALKS / 'summary.json')
patterns = pd.read_csv(OUT / 'answer_metapaths.csv')
rules = read(OUT / 'rule_catalog.json')

def table(frame):
    markup = frame.to_html(index=False,escape=True).replace(r'\n', '<br>')
    display(HTML(markup.replace('<table ', '<table style="text-align:left" ')))

rows = []
for key, name in [('gandalf_walks','Gandalf walks'),('aragorn_rules','Local ARAGORN')]:
    v = comparison['results'][key][SPLIT]
    rows.append({'Method':name, 'Recovered / expected':f"{v['answers_reached']:,} / {v['answer_pairs']:,}",
                 'Recovery':f"{v['answer_coverage']:.2%}", 'Mean graph connections':round(v['graph_sizes']['edges']['mean'])})
display(Markdown(f'**{SPLIT.title()} comparison**'))
table(pd.DataFrame(rows))


**Validation comparison**

Method,Recovered / expected,Recovery,Mean graph connections
Gandalf walks,"1,533 / 2,291",66.91%,2101
Local ARAGORN,"455 / 2,291",19.86%,217


## Gandalf: top observed metapaths

No fixed rules: these are observed 1–3 step prefixes of completed three-edge walks. Arrows preserve stored edge direction; categories are descriptive and qualifiers are not used by the sampler.


In [3]:
top = patterns[patterns.split.eq(SPLIT)].sort_values(
    ['distinct_answer_pairs','metapath'],ascending=[False,True]).head(TOP_N)
table(top[['metapath','distinct_answer_pairs']].rename(columns={
    'metapath':'Observed metapath', 'distinct_answer_pairs':'Expected pairs recovered'}))


Observed metapath,Expected pairs recovered
Disease <-[in_clinical_trials_for]- ChemicalEntity,786
Disease <-[treats_or_applied_or_studied_to_treat]- ChemicalEntity,534
Disease <-[applied_to_treat]- ChemicalEntity,361
Disease <-[in_clinical_trials_for]- ChemicalEntity -[in_clinical_trials_for]-> Disease <-[in_clinical_trials_for]- ChemicalEntity,123
Disease <-[treats]- ChemicalEntity -[treats]-> Disease <-[treats]- ChemicalEntity,84


## Local ARAGORN: top rule patterns

Every listed edge and qualifier must match together; some rules are triangles rather than single paths. **A = answer chemical; D = query disease/phenotype.** Intermediate node types are shown in each row. Brackets give object aspect/direction qualifiers; query predicates can expand through Biolink.


In [4]:
short = lambda s: s.removeprefix('biolink:')
alias = lambda n: {'$source':'A','$target':'D'}.get(n,n.upper())

def motif(q):
    lines = []
    for e in q['edges'].values():
        p = '|'.join(map(short,e['predicates']))
        qualifiers = e.get('constraints',{}).get('qualifiers',[])
        if qualifiers:
            p += ' [' + ' OR '.join(', '.join(
                (str(v) if k in ('biolink:object_aspect_qualifier','biolink:object_direction_qualifier')
                 else f'{short(k)}={v}') for k,v in qc.items()) for qc in qualifiers) + ']'
        lines.append(f"{alias(e['subject'])} —{p}→ {alias(e['object'])}")
    return '\n'.join(lines)

ranked = sorted((r for r in rules if r[SPLIT]['answers_bound']>0),
                key=lambda r:(-r[SPLIT]['answers_bound'],r['rule_id']))[:TOP_N]
rows = []
for r in ranked:
    q = r['query_graph']
    rows.append({'Rule':r['rule_id'], 'Required pattern':motif(q),
                 'Intermediate types':'; '.join(f"{alias(n)}: {'|'.join(map(short,v['categories']))}"
                                               for n,v in q['nodes'].items() if n not in ('$source','$target')),
                 'Expected pairs recovered':r[SPLIT]['answers_bound']})
table(pd.DataFrame(rows))
assert len(rules)==18


Rule,Required pattern,Intermediate types,Expected pairs recovered
5,A —treats_or_applied_or_studied_to_treat→ DA —treats_or_applied_or_studied_to_treat→ HH —has_phenotype→ D,H: Disease,421
9,A —treats_or_applied_or_studied_to_treat→ FF —has_phenotype→ DD —has_phenotype→ F,F: Disease,109
10,"E —treats→ DI —affects [activity_or_abundance, decreased]→ AI —affects [activity_or_abundance, decreased]→ E",E: ChemicalEntity; I: BiologicalEntity,31
13,F —positively_correlated_with→ DA —treats→ F,F: Disease,14
16,"E —treats→ DI —affects [activity, increased]→ AI —affects [activity_or_abundance, increased]→ E",E: ChemicalEntity; I: BiologicalEntity,14


**What differs:** Gandalf can recover answers through a single retained treatment-related edge. ARAGORN requires a complete rule pattern—for example, rule 5 also needs a second disease and a `has_phenotype` link. Gandalf's larger retrieved graphs also affect recovery.

[Methods, verification, and full cached results](../pilot/metapath_analysis/README.md)
